## Imports

In [ ]:
print("In file, starting imports..............")
import os
import sys
from pathlib import Path
import torch
from torch.utils.data import DataLoader
import json
import pandas as pd
import numpy as np

In [ ]:
def find_repo_root(marker="data") -> Path:
    """Walk up directory tree until we find a folder containing `marker`."""
    current = Path().resolve()
    for parent in [current, *current.parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"Could not find repo root (looked for '{marker}' folder)")

PAR_DIR = find_repo_root(marker="README.md")
if str(PAR_DIR) not in sys.path:
    sys.path.append(str(PAR_DIR))

DATASET_PATH = PAR_DIR / "data"

In [ ]:
from src.deter_architecture import Unet
from src.dataset import CustomPTDataset, create_train_test, get_seasons
from src.plots import  SaliencyPlotPaper

## Load data

In [ ]:
ALL_YEARS = [2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]
HELD_OUT_YEARS = [2008, 2015, 2023]
ALL_LOYO_YEARS = [year for year in ALL_YEARS if year not in HELD_OUT_YEARS]
TRAIN_YEARS = ALL_LOYO_YEARS
TEST_YEARS = HELD_OUT_YEARS
SCALER_PATH = os.path.join(DATASET_PATH, f"scaler", "scaler_full.pkl")

print("Creating the dataset and dataloader..............")
_, _, TEST_DATASET = create_train_test(DATASET_PATH, TRAIN_YEARS, TEST_YEARS, scaler_path=SCALER_PATH)
TEST_DATASET.metadata_csv["year"] = pd.to_datetime(TEST_DATASET.metadata_csv["date"]).dt.year

extreme_days = pd.read_csv(DATASET_PATH / "extreme_days_top_0.05.csv")
all_extremes_metadata = TEST_DATASET.metadata_csv[TEST_DATASET.metadata_csv["date"].isin(extreme_days["date"])]
test_extremes = all_extremes_metadata[all_extremes_metadata["year"].isin(TEST_YEARS)]
test_extremes_ids = test_extremes["id"].values
TEST_EXTREMES_DATASET = CustomPTDataset(root_dir=DATASET_PATH, sample_ids=test_extremes_ids, scaler_path=SCALER_PATH)

TEST_EXTREMES_LOADER = DataLoader(TEST_EXTREMES_DATASET, batch_size=1, shuffle=False)

ALL_DAYS_SEASONS = np.array(get_seasons(ALL_YEARS))
EXTREMES_SEASONS = ALL_DAYS_SEASONS[test_extremes_ids]

## Load models

In [ ]:
MODELS_PATH = Path(PAR_DIR) / "models"

CHANNELS = [16, 32, 64]
NUM_RESIDUAL_LAYERS = 2
RECALIBRATION = 'platt_scaling'

# Initialize unet
unet = Unet(
    channels = CHANNELS,
    num_residual_layers = NUM_RESIDUAL_LAYERS,
    name = "unet",
    recalibration_method = RECALIBRATION,
)

unet.load_state_dict(torch.load(MODELS_PATH / 'unet.pth', map_location=torch.device('cpu')))
unet.eval()

## Saliency plot

In [ ]:
test_extremes.head()

In [ ]:
event_id = 32
day = test_extremes.iloc[event_id]["date"]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
extremes_batch = TEST_DATASET[event_id]

In [ ]:
lat = 40
lon = 16
hour = 18
time = pd.to_datetime(day) + pd.Timedelta(f"{hour}:00:00")

In [ ]:

saliency = SaliencyPlotPaper(extremes_batch[hour:hour+1, :-1], unet, TEST_DATASET.metadata_json, lat, lon, 
                        f"Saliency map on {time.strftime('%Y-%m-%d')} at {time.strftime('%H:%M')} at location {lat}N, {lon}E", 
                        device, save_path=None, file_name=None)